# Project 2: Enterprise Hybrid RAG Knowledge Assistant
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![Google GenAI SDK](https://img.shields.io/badge/SDK-google--genai%20v1.0+-blue.svg)](https://github.com/googleapis/python-genai)
[![Models](https://img.shields.io/badge/Models-Gemini%202.5%20%7C%20text--embedding--004-orange.svg)](https://ai.google.dev)

---

## 🌟 Welcome to Project 2: Master RAG from the Ground Up!
Have you ever asked an AI model a question about your company's private internal documentation, only for it to make up a convincing but completely false answer? That is called **hallucination**.

In this project, you will learn the single most important technique in enterprise AI: **Retrieval-Augmented Generation (RAG)**.

---

## 🏛️ Enterprise Hybrid RAG System Architecture

<p align="center">
  <img src="./images/02_enterprise_rag_architecture.png" alt="02_module_learning_roadmap" width="100%" />
</p>

---

## 🗺️ What You Will Master in This Project:
<p align="center">
  <img src="./images/02_module_learning_roadmap.png" alt="02_enterprise_rag_architecture" width="100%" />
</p>


---
## Environment Setup & Client Initialization
We install `google-genai`, `chromadb`, `pydantic`, `rank-bm25`, `scikit-learn`, and visualization libraries.


In [ ]:
# Install required packages for RAG and Vector Search
!pip install -q google-genai chromadb pydantic rank-bm25 scikit-learn rich


In [ ]:
import os
import getpass
from google import genai
from google.genai import types

# Authenticate via API Key or Vertex AI
if "GEMINI_API_KEY" not in os.environ:
    api_key = getpass.getpass("Enter your Gemini API Key (or press enter for Vertex AI auth): ")
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key

if os.environ.get("GEMINI_API_KEY"):
    client = genai.Client()
    print("✅ Google GenAI Client initialized via API Key.")
else:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
    LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
    print(f"✅ Google GenAI Client initialized via Vertex AI ({PROJECT_ID}).")


---
## 1. Overview of Model Adaptation Techniques: Choosing the Right Tool

When you want an AI model to solve a specific company problem, you have three main tools in your engineering toolbox:

<p align="center">
  <img src="./images/02_adaptation_spectrum_rag_lora.png" alt="02_adaptation_spectrum_rag_lora" width="100%" />
</p>

---

### 📊 When to Use Which Technique:

| Technique | Cost to Implement | Knowledge Updates | Verifiability (Citations) | Best Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Prompting** | Free / Seconds | Static (per prompt) | Manual | Quick experiments, simple instructions, formatting |
| **RAG (This Lab)** | Low / Minutes | **Real-Time** (instant DB insert) | **100% Verifiable** (exact chunk links) | Enterprise knowledge bases, policy documents, support bots |
| **Fine-Tuning (LoRA)** | Moderate / Hours | Slow (requires re-training) | Opaque (embedded in weights) | Teaching specialized domain vocabularies, SQL dialects |


---
## 2. Advanced Prompt Engineering & Structured Outputs

Prompt engineering is the software interface to the LLM. Modern enterprise systems enforce strict schema validation using **Pydantic** and `response_schema`.

Let's demonstrate structured output extraction directly with `google-genai` and Gemini 2.5 Flash:


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define strict Pydantic response schema
class SupportTicketAnalysis(BaseModel):
    customer_intent: str = Field(description="Primary category: Billing, Technical, Account, Feature_Request")
    urgency_level: str = Field(description="Low, Medium, High, Critical")
    sentiment_score: float = Field(description="Sentiment score from -1.0 to 1.0")
    actionable_steps: List[str] = Field(description="Concrete resolution steps for the tier-2 engineer")
    sla_breach_risk: bool = Field(description="True if customer mentions downtime or financial impact")

# Test Ticket Ingestion
customer_message = (
    "URGENT: Our production Cloud Run service in europe-west1 is failing health checks with HTTP 503 errors "
    "after we updated the container image 20 minutes ago. Over 5,000 customers are affected and transactions "
    "are dropping. We need an engineer immediately!"
)

# Call Gemini with Structured Output Schema
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"Analyze the following enterprise support ticket:\n{customer_message}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=SupportTicketAnalysis,
        temperature=0.0
    )
)

print("📋 Structured Output Extracted by Gemini:")
print(response.text)


---
## 3. RAG Architecture: Ingestion, Chunking & Embeddings

<p align="center">
  <img src="./images/02_ingestion_semantic_chunking_pipeline.png" alt="02_ingestion_semantic_chunking_pipeline" width="100%" />
</p>

### 📍 Embeddings Explained: "GPS Coordinates for Concepts"
Think of an embedding model as a machine that places every idea onto a map:
- `"Dog"` and `"Puppy"` have coordinates very close to each other.
- `"King"` minus `"Man"` plus `"Woman"` lands right at the coordinate for `"Queen"`.
- Cosine similarity measures the angle between these vectors: **$1.0$ = Identical meaning**, **$0.0$ = Unrelated**.


In [ ]:
# Generating Dense Embeddings using text-embedding-004

documents = [
    "Google Cloud Run automatically scales containers from zero to thousands of instances based on incoming request traffic.",
    "Vertex AI Vector Search provides high-throughput, millisecond-latency approximate nearest neighbor search over billions of vectors.",
    "Cloud Billing Budgets and Alerts send programmatic Pub/Sub notifications when project spend reaches 50%, 80%, or 100% of target.",
    "BigQuery is a fully managed, serverless enterprise data warehouse that enables SQL queries over petabytes of structured data."
]

embed_response = client.models.embed_content(
    model="text-embedding-004",
    contents=documents
)

print(f"✅ Generated {len(embed_response.embeddings)} embeddings.")
print(f"📐 Vector Dimensionality: {len(embed_response.embeddings[0].values)} dimensions.")
print(f"🔍 Sample vector values: {embed_response.embeddings[0].values[:5]}...")


---
## 4. Hybrid Search: Combining Dense Semantic & Sparse Keyword Retrieval

In enterprise production systems, pure vector search has a dangerous blind spot:
- **Vector search** is amazing at understanding general concepts (*"How do I cancel my subscription?"*).
- But **Vector search often fails** at exact alphanumeric codes (*"Error Code 503-AUTH-TIMEOUT"*, part number `"SKU-992-NVDA"`, or function name `handle_auth_callback()`).

### 🏆 The Industry Standard: Hybrid Search + Reciprocal Rank Fusion (RRF)
To build a system that never fails, forward deployment engineers run **two search engines in parallel**:

<p align="center">
  <img src="./images/02_hybrid_search_rrf_pipeline.png" alt="02_hybrid_search_rrf_pipeline" width="100%" />
</p>


---
## 5. Enterprise Reliability: Handling Noisy & Irrelevant Search Results (RAFT Pattern)

In real enterprise deployments, your search engine will occasionally return noisy or irrelevant documents alongside the correct ones.

<p align="center">
  <img src="./images/02_raft_distractor_defense_shield.png" alt="02_raft_distractor_defense_shield" width="100%" />
</p>


---
## 6. End-to-End Implementation: Enterprise Customer Support RAG System

Let's build a complete, production-ready Customer Support Knowledge System with ChromaDB, BM25, and Gemini 2.5 Flash!


In [ ]:
import chromadb
from rank_bm25 import BM25Okapi
import numpy as np
from typing import List, Dict, Any

# 1. Curate Enterprise Customer Support Knowledge Base
KNOWLEDGE_BASE = [
    {
        "id": "KB-101",
        "title": "Cloud Run Health Check & 503 Troubleshooting",
        "content": "Cloud Run instances return HTTP 503 (Service Unavailable) when container startup times exceed the configured startupProbe timeout (default 240s), or when the container crashes immediately due to an unhandled SIGTERM. To resolve: check Cloud Logging for container panic logs, ensure the container listens on the PORT environment variable (default 8080), and verify memory allocation matches workload requirements."
    },
    {
        "id": "KB-102",
        "title": "Enterprise Cloud SLA & Financial Credit Policy",
        "content": "Google Cloud Run provides a Monthly Uptime Percentage SLA of 99.95%. If the Monthly Uptime drops between 99.0% and <99.95%, customers are entitled to a 10% Financial Credit. If uptime drops below 95.0%, a 50% credit applies. Claims must be submitted to Cloud Support within 30 days of the incident with incident IDs and error log timestamps."
    },
    {
        "id": "KB-103",
        "title": "Vertex AI Vector Search Index Deployment Guidelines",
        "content": "When deploying a Vertex AI Vector Search index endpoint, configure minimum replica count >= 2 across distinct zones to satisfy high availability requirements. For HNSW indexes, set distanceMeasureType to COSINE_DISTANCE or DOT_PRODUCT_DISTANCE. Query latency target is <10ms for p95 at 1,000 QPS."
    },
    {
        "id": "KB-104",
        "title": "IAM Zero-Trust Principle for Cloud Run Service-to-Service",
        "content": "To enable secure service-to-service communication on Cloud Run without public ingress, set ingress settings to 'Internal and Cloud Load Balancing', grant the calling service account the 'roles/run.invoker' IAM role on the receiving service, and pass an OpenID Connect (OIDC) identity token in the Authorization header: Authorization: Bearer $(gcloud auth print-identity-token)."
    },
    {
        "id": "KB-105",
        "title": "Cloud Billing Budget Alerts & Programmatic Caps",
        "content": "Cloud Billing Budgets trigger automated alerts at 50%, 90%, and 100% of forecast or actual spend. To programmatically disable billing or stop VM instances upon budget exhaustion, connect the budget to a Cloud Pub/Sub topic and trigger a Cloud Run or Cloud Function service that calls Google Cloud Billing APIs to disable project billing."
    }
]

print(f"📚 Loaded {len(KNOWLEDGE_BASE)} enterprise knowledge base documents.")


In [ ]:
class EnterpriseHybridRAG:
    """Production Hybrid RAG Engine integrating text-embedding-004 and BM25 with RRF."""
    
    def __init__(self, client: genai.Client, documents: List[Dict[str, str]]):
        self.client = client
        self.docs = documents
        
        # 1. Initialize Vector Database (ChromaDB In-Memory)
        self.chroma_client = chromadb.Client()
        try:
            self.chroma_client.delete_collection("support_kb")
        except:
            pass
        self.collection = self.chroma_client.create_collection(
            name="support_kb",
            metadata={"hnsw:space": "cosine"}
        )
        
        # 2. Build BM25 Inverted Index
        self.corpus = [doc["title"] + " " + doc["content"] for doc in documents]
        self.tokenized_corpus = [text.lower().split() for text in self.corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        
        # 3. Embed and Index into Vector DB
        self._index_documents()
        
    def _index_documents(self):
        print("⚙️ Indexing documents with text-embedding-004...")
        embeddings_resp = self.client.models.embed_content(
            model="text-embedding-004",
            contents=self.corpus
        )
        
        vectors = [emb.values for emb in embeddings_resp.embeddings]
        ids = [doc["id"] for doc in self.docs]
        metadatas = [{"title": doc["title"]} for doc in self.docs]
        
        self.collection.add(
            ids=ids,
            embeddings=vectors,
            documents=self.corpus,
            metadatas=metadatas
        )
        print(f"✅ Indexed {len(ids)} documents into ChromaDB Vector Index.")
        
    def hybrid_search(self, query: str, top_k: int = 3, rrf_k: int = 60) -> List[Dict[str, Any]]:
        """Executes Dense + Sparse search fused via Reciprocal Rank Fusion."""
        # A. Dense Vector Retrieval
        query_emb = self.client.models.embed_content(
            model="text-embedding-004",
            contents=query
        ).embeddings[0].values
        
        vector_results = self.collection.query(
            query_embeddings=[query_emb],
            n_results=len(self.docs)
        )
        dense_ranked_ids = vector_results["ids"][0]
        
        # B. Sparse BM25 Retrieval
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        sparse_ranked_indices = np.argsort(bm25_scores)[::-1]
        sparse_ranked_ids = [self.docs[i]["id"] for i in sparse_ranked_indices]
        
        # C. Reciprocal Rank Fusion (RRF)
        rrf_scores = {doc["id"]: 0.0 for doc in self.docs}
        
        # Add dense ranks
        for rank, doc_id in enumerate(dense_ranked_ids, start=1):
            rrf_scores[doc_id] += 1.0 / (rrf_k + rank)
            
        # Add sparse ranks
        for rank, doc_id in enumerate(sparse_ranked_ids, start=1):
            rrf_scores[doc_id] += 1.0 / (rrf_k + rank)
            
        # Sort by final fused RRF score
        sorted_doc_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        
        # Construct retrieved records
        doc_lookup = {d["id"]: d for d in self.docs}
        retrieved = []
        for doc_id, score in sorted_doc_ids:
            item = doc_lookup[doc_id].copy()
            item["rrf_score"] = round(score, 5)
            retrieved.append(item)
            
        return retrieved

    def generate_grounded_response(self, customer_query: str) -> str:
        """Retrieves context and synthesizes grounded answer with citations."""
        retrieved_docs = self.hybrid_search(customer_query, top_k=2)
        
        # Build strict grounded context block
        context_str = "\n\n".join([
            f"[Document ID: {d['id']} | Title: {d['title']}]\n{d['content']}"
            for d in retrieved_docs
        ])
        
        system_prompt = """
You are an expert Google Cloud Tier-3 Enterprise Support Engineer.
Rules for your response:
1. Answer the customer's question strictly and exclusively using the provided Knowledge Base Documents.
2. Every factual claim MUST include an inline citation in the format [KB-XXX].
3. If the answer cannot be found in the provided context, explicitly state: "Based on our official enterprise documentation, this information is not available. Please contact your dedicated Customer Engineer."
4. Do not speculate or invent configuration parameters.
"""

        user_prompt = f"""
KNOWLEDGE BASE CONTEXT:
==================================================
{context_str}
==================================================

CUSTOMER INQUIRY:
{customer_query}
"""

        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.0
            )
        )
        return response.text

# Initialize Engine
rag_engine = EnterpriseHybridRAG(client, KNOWLEDGE_BASE)
print("🚀 EnterpriseHybridRAG initialized and ready.")


In [ ]:
# Test RAG Query
test_query = "Our Cloud Run service is returning 503 errors and we had 2 hours of downtime. What caused it and what SLA credit can we claim?"

print(f"❓ Customer Query: {test_query}\n")
answer = rag_engine.generate_grounded_response(test_query)
print("="*40 + " [GROUNDED GEMINI RESPONSE] " + "="*40)
print(answer)


---
## 7. Practical Hands-On Exercises (With Beginner Hints & Starter Code)

Test your applied Applied AI engineering skills by completing these three enterprise challenges:

---

### 🎯 Exercise 2.1: Semantic Chunking Engine
**The Business Scenario**: Traditional chunking cuts text every 500 characters, often splitting key ideas in half. Semantic chunking groups sentences together until the topic changes!
- **Task**: Implement `semantic_chunk_text(text: str, client: genai.Client)` to calculate distance between consecutive sentences and split when the topic changes.
- **💡 Beginner Hint**: Generate embeddings for each sentence using `client.models.embed_content(model="text-embedding-004", ...)`. When consecutive sentences have low similarity, create a new chunk!

---

### 🎯 Exercise 2.2: Multi-Query Search Expander
**The Business Scenario**: Users often type short, ambiguous queries like *"503 broken"*. A forward deployment engineer expands this into 3 rich queries to maximize search recall.
- **Task**: Implement `expand_query(query: str, client: genai.Client) -> QueryExpansionResult` using Gemini with structured Pydantic output.
- **💡 Beginner Hint**: Ask Gemini to generate technical synonyms (e.g. *"Cloud Run HTTP 503 service unavailable"*, *"Container instance crash remediation"*).

---

### 🎯 Exercise 2.3: Automated Faithfulness Evaluator (LLM-as-a-Judge)
**The Business Scenario**: Before shipping a customer chatbot to production, you need an automated test suite that checks if the chatbot's answers are 100% grounded in company docs.
- **Task**: Implement `evaluate_rag_faithfulness(question: str, context: str, answer: str, client: genai.Client)` to score groundedness from 0.0 to 1.0 and flag any hallucinated sentences.
- **💡 Beginner Hint**: Use a Pydantic schema with `is_faithful: bool`, `groundedness_score: float`, and `hallucinations: List[str]`.


---
## 8. Step-by-Step Solutions & Architectural Explanations


In [ ]:
# ============================================================================
# SOLUTION 2.1: Semantic Chunking Engine
# ============================================================================
import re
from sklearn.metrics.pairwise import cosine_similarity

def semantic_chunk_text(
    text: str,
    client: genai.Client,
    distance_threshold_percentile: float = 80.0
) -> List[str]:
    """
    Splits text dynamically based on statistical semantic distance shifts.
    """
    # Split text into sentences using regex
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]
    if len(sentences) <= 1:
        return [text]
        
    # Generate embeddings for each sentence
    embed_resp = client.models.embed_content(
        model="text-embedding-004",
        contents=sentences
    )
    embeddings = np.array([e.values for e in embed_resp.embeddings])
    
    # Compute cosine distances between adjacent sentences
    distances = []
    for i in range(len(embeddings) - 1):
        cos_sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        distances.append(1.0 - cos_sim)
        
    distances = np.array(distances)
    # Calculate threshold based on percentile
    threshold = np.percentile(distances, distance_threshold_percentile)
    
    # Group sentences into semantic chunks
    chunks = []
    current_chunk = [sentences[0]]
    for i, dist in enumerate(distances):
        if dist > threshold:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentences[i+1]]
        else:
            current_chunk.append(sentences[i+1])
            
    if current_chunk:
        chunks.append(" ".join(current_chunk))
        
    return chunks

# Test Semantic Chunking
sample_article = (
    "Google Cloud Spanner is a globally distributed database that provides ACID transactions with external consistency. "
    "It utilizes TrueTime API backed by atomic clocks and GPS receivers to assign monotonically increasing timestamps. "
    "In contrast, Google Cloud Pub/Sub is a globally distributed asynchronous messaging backbone. "
    "It provides at-least-once message delivery with sub-100ms latency. Subscribers pull messages or receive webhooks via push endpoints. "
    "For container workloads, Google Kubernetes Engine (GKE) provides automated cluster management, autoscaling node pools, and deep integration with Cloud VPC networking."
)

chunks = semantic_chunk_text(sample_article, client, distance_threshold_percentile=70.0)
print(f"✂️ Extracted {len(chunks)} semantic chunks:")
for idx, ch in enumerate(chunks, 1):
    print(f"\n--- [Chunk {idx}] ---")
    print(ch)


In [ ]:
# ============================================================================
# SOLUTION 2.2: Query Expansion & Reformulation
# ============================================================================
class QueryExpansionResult(BaseModel):
    expanded_queries: List[str] = Field(description="3 distinct search query reformulations with technical terminology")
    key_entities: List[str] = Field(description="Identified cloud services, error codes, and technical entities")

def expand_query(query: str, client: genai.Client) -> QueryExpansionResult:
    prompt = f"""
Analyze the customer search query and generate 3 diverse query reformulations optimized for dense vector and keyword search:
Query: "{query}"
"""
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=QueryExpansionResult,
            temperature=0.2
        )
    )
    return QueryExpansionResult.model_validate_json(resp.text)

expanded = expand_query("my website is down on gcp container", client)
print("🔍 Expanded Queries for Hybrid Search:")
for q in expanded.expanded_queries:
    print(f" • {q}")
print(f"🏷️ Key Entities: {expanded.key_entities}")


In [ ]:
# ============================================================================
# SOLUTION 2.3: Automated LLM-as-a-Judge Faithfulness Evaluator
# ============================================================================
class RAGEvaluationReport(BaseModel):
    faithfulness_score: float = Field(description="Score between 0.0 and 1.0 indicating if all claims are grounded in context")
    context_relevance_score: float = Field(description="Score between 0.0 and 1.0 indicating if context was sufficient")
    unsupported_claims: List[str] = Field(description="List of any claims made in the answer not supported by the context")
    evaluation_rationale: str = Field(description="Detailed explanation of the judge's scoring decisions")

def evaluate_rag_faithfulness(
    question: str,
    context: str,
    answer: str,
    client: genai.Client
) -> RAGEvaluationReport:
    judge_prompt = f"""
You are an impartial AI Quality and Safety Auditor evaluating an enterprise RAG system.
Evaluate the following Interaction:

[QUESTION]:
{question}

[RETRIEVED CONTEXT]:
{context}

[GENERATED ANSWER]:
{answer}

Carefully verify if every single statement in the answer is directly supported by the context.
"""
    resp = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=judge_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RAGEvaluationReport,
            temperature=0.0
        )
    )
    return RAGEvaluationReport.model_validate_json(resp.text)

# Run Evaluation on our RAG Answer
sample_context = KNOWLEDGE_BASE[0]["content"] + "\n" + KNOWLEDGE_BASE[1]["content"]
eval_report = evaluate_rag_faithfulness(test_query, sample_context, answer, client)

print("⚖️ RAG LLM-as-a-Judge Evaluation Report:")
print(f"• Faithfulness Score: {eval_report.faithfulness_score * 100:.1f}%")
print(f"• Context Relevance:  {eval_report.context_relevance_score * 100:.1f}%")
print(f"• Unsupported Claims: {eval_report.unsupported_claims if eval_report.unsupported_claims else 'None (Fully Grounded)'}")
print(f"• Rationale: {eval_report.evaluation_rationale}")


---
## Summary & Key Takeaways
1. **Hybrid Retrieval**: Combining dense vectors (`text-embedding-004`) with sparse BM25 via Reciprocal Rank Fusion overcomes vocabulary mismatch while preserving deep semantic retrieval.
2. **Strict Grounding**: Structured prompt templates with citation constraints prevent hallucinations in enterprise customer support environments.
3. **Automated Evaluation**: LLM-as-a-Judge frameworks evaluate faithfulness and context relevance automatically before deploying updates to production.
4. **Next Step**: In **Project 3**, we evolve from single-turn retrieval to autonomous multi-step agents with **Tool Calling, Workflows, MCP, and building an "Ask-the-Web" Perplexity engine**!
